# Реранкинг: LightGBM заново упорядочивает кандидатов бейзлайна

**Данные.** Корпус — 189 212 объявлений из `benchmark_items.parquet`: только среди них выбираем 50 ответов на тестовый запрос. Тестовые запросы — 2 452 запроса из `benchmark_queries.parquet`, ответов к ним у нас нет: качество ответа на них считают организаторы и показывают на лидерборде. Для обучения есть train — 497 673 строки «поля поиска → выбранное объявление». Номера запроса в train нет: запросом считаем набор полей поиска (текст, локация, фильтры, категория, флаг доставки), и строки с одинаковыми полями склеиваем в один запрос. Выбранные в train объявления в основном в корпус не входят: объявление из корпуса выбрано только в 6.6% строк train (`validation/split.ipynb`). Поэтому в общей таблице объявлений есть пометка `in_corpus` — входит ли объявление в корпус.

**Метрика.** Recall@50: для каждого запроса считаем, какая доля его правильных объявлений (тех, что выбрал пользователь) попала в первые 50 ответа, и усредняем по запросам. Recall@N — то же для первых N. Разницу двух значений Recall пишем в процентных пунктах (п.п.): рост с 0.885 до 0.907 — это +2.2 п.п.

**Как проверяем.** Валидация — 5 032 проверочных запроса из train с известными ответами, собранные так, чтобы быть похожими на тестовые (`validation/split.ipynb`). «Валидация» и «проверочные запросы» дальше означают одно и то же.

**Зачем второй этап.** Бейзлайн (`retrieval/baseline.ipynb`) — первое решение, без обучения. Для каждой пары «запрос — объявление» он считает оценку BM25: она тем выше, чем больше слов запроса нашлось в объявлении, чем реже эти слова встречаются в других объявлениях корпуса и чем чаще они повторяются в этом объявлении. К оценке прибавляется поправка за локацию (раздел 2). Recall@50 бейзлайна — 0.885, а в его первые 500 объявлений попадает 96.9% правильных (раздел 1). То есть в первые 50 не попадает 11.5% правильных объявлений, и большую часть из них — 8.4% всех правильных — бейзлайн всё же находит, но ставит на места с 51-го по 500-е. Их и можно поднять. Поэтому решение работает в два этапа:

1. **Отбор кандидатов.** Первый этап — ретривер; здесь это сам бейзлайн. Он берёт для каждого запроса 500 объявлений с лучшей оценкой.
2. **Реранкинг.** Для каждой пары «запрос — кандидат» считаем набор чисел (признаков): оценки BM25 по разным полям объявления, расстояние до места поиска, подкатегорию и так далее. Модель LightGBM — реранкер — по этим признакам заново упорядочивает 500 кандидатов, и первые 50 идут в ответ.

**Как учится LightGBM.** Модель строит одно за другим много небольших деревьев решений, каждое исправляет ошибки предыдущих. В режиме LambdaRank она учится так, чтобы внутри одного запроса правильные объявления стояли выше неправильных, причём ошибки в начале списка штрафуются сильнее, чем в конце.

**На чём учится.** Train поделён на три непересекающиеся части (`validation/split.ipynb`): валидация, обучение реранкера (6 487 запросов) и обучение ретривера (остальные 482 161 строка). Реранкер учится на запросах для обучения реранкера, а качество меряем на валидации. Признаки локации опираются на доли выборов по локациям: для каждой локации поиска — в каких локациях и как часто выбирали объявления в train. Эти доли считаются только по обучению ретривера. Если бы в них попали строки запросов реранкера, правильный ответ каждого такого запроса уже был бы учтён в долях, и модель научилась бы доверять им больше, чем они того стоят на новых запросах. 10% запросов реранкера откладываем, чтобы решить, когда перестать добавлять деревья (раздел 3).

Кандидатов и признаки считает `scripts/ranking/candidates.py` с флагом `--bm25-only`: кандидаты берутся только из бейзлайна, без поиска по смыслу, который добавляется в `ranking/dense.ipynb`. Код — `avito.ranking`.

In [1]:
import polars as pl

from avito import paths
from avito.ingestion import load as ingested
from avito.metrics.recall import bootstrap_interval, query_recalls
from avito.ranking.features import FEATURES, ITEM_FEATURES, RANKING_FEATURES
from avito.ranking.model import PARAMS, fit_ranker, rerank
from avito.validation.load import load_val_queries

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(220);

Загружаем кандидатов с уже посчитанными признаками: одна строка — одна пара «запрос — кандидат», по 500 кандидатов на запрос. Колонка `label` равна 1, если кандидат — правильное объявление запроса.

In [2]:
ranker_candidates = pl.read_parquet(paths.candidates_path("ranker", None))
val_candidates = pl.read_parquet(paths.candidates_path("val", None))
val = load_val_queries()

pl.DataFrame({
    "набор": ["обучение реранкера", "валидация"],
    "запросов": [ranker_candidates["query_id"].n_unique(), val_candidates["query_id"].n_unique()],
    "пар «запрос — кандидат»": [ranker_candidates.height, val_candidates.height],
})

набор,запросов,пар «запрос — кандидат»
str,i64,i64
"""обучение реранкера""",6487,3243500
"""валидация""",5032,2516000


## 1. Сколько правильных объявлений среди кандидатов

Считаем Recall@N бейзлайна для N = 50, 100, 200, 300 и 500: какая доля правильных объявлений попала в его первые N. Реранкер только переставляет 500 кандидатов, поэтому Recall@500 — предел, выше которого он не поднимется.

In [3]:
def top_by_rank(candidates: pl.DataFrame, depth: int) -> pl.DataFrame:
    top = candidates.filter(pl.col("retrieval_rank") < depth)
    return top.group_by("query_id", maintain_order=True).agg("item_id")


depths = [50, 100, 200, 300, 500]
pl.DataFrame({
    "сколько первых объявлений бейзлайна берём": depths,
    "Recall на этой глубине": [query_recalls(top_by_rank(val_candidates, depth), val, k=depth).mean() for depth in depths],
})

сколько первых объявлений бейзлайна берём,Recall на этой глубине
i64,f64
50,0.885135
100,0.919118
200,0.947271
300,0.956677
500,0.9687


Recall@500 = 0.969, Recall@50 = 0.885. Значит, реранкер может поднять Recall@50 самое большее на 8.4 п.п.: с 0.885 до 0.969. Остальные 3.1% правильных объявлений в 500 кандидатов не попадают, и их реранкер не достанет.

## 2. Признаки

Все признаки считаются для каждой пары «запрос — кандидат» (`avito.ranking.candidates`, `avito.ranking.features`).

| Группа | Признак | Что это |
|---|---|---|
| позиция в бейзлайне | `retrieval_rank` | место кандидата в выдаче бейзлайна; места считаются с нуля: 0 — первое, 499 — последнее |
| | `retrieval_score` | оценка, по которой бейзлайн сортирует объявления. Это BM25 по тексту запроса плюс 0.3 · BM25 по тексту фильтров, делённое на наибольшее такое значение среди объявлений корпуса для этого запроса (лучшее по словам получает 1), плюс поправка за локацию `0.1 · log(доля выборов + 0.0003)`. Доля выборов — как часто при поиске в локации запроса выбирали объявления из локации кандидата. Оба веса взяты из бейзлайна. Добавка 0.0003 нужна, чтобы объявления из локаций, где при таком поиске ни разу не выбирали, сильно опускались, но не выбрасывались совсем |
| текст | `bm25_full` | BM25 по всему объявлению: заголовок, параметры и описание склеены в один текст, заголовок повторён трижды, чтобы его слова весили больше, чем слова длинного описания (как в бейзлайне) |
| | `bm25_title`, `bm25_params`, `bm25_description` | BM25 отдельно по заголовку, параметрам и описанию |
| | `bm25_filters` | BM25 текста фильтров запроса («Вид услуги Красота, здоровье») по всему объявлению |
| | `coverage_full`, `coverage_title`, `coverage_params`, `coverage_description` | какая доля слов запроса есть во всём объявлении и в каждом поле |
| | `bm25_full_relative` | `bm25_full`, делённый на максимальный `bm25_full` среди кандидатов запроса |
| | `query_max_bm25` | максимальный `bm25_full` среди кандидатов запроса: есть ли у запроса вообще хорошие совпадения по словам |
| локация | `log_prior` | та же поправка за локацию, что в `retrieval_score`, отдельным признаком: `log(доля выборов + 0.0003)` |
| | `same_location` | лежит ли объявление в той же локации, где искали |
| | `distance_km` | расстояние в километрах от объявления до центра локации поиска; центр — медиана координат объявлений, которые выбирали при поиске в этой локации (считается по той же части train, что и доли выборов) |
| | `same_location_share` | какая доля кандидатов запроса лежит в локации поиска: много ли вообще подходящих объявлений рядом |
| подкатегория | `microcat_share` | Подкатегория — самый мелкий раздел каталога Авито, к которому относится объявление (в данных это номер `item_microcat_id`). Признак — какая доля первых 30 кандидатов бейзлайна относится к той же подкатегории, что и этот кандидат. Если 20 из 30 первых — «Маникюр», запрос, скорее всего, про маникюр, и кандидат из «Маникюра» получает 20 / 30 = 0.67. Число 30 мы не подбирали |
| запрос | `query_terms` | сколько разных слов в запросе |
| | `has_filters` | есть ли у запроса фильтры |
| | `search_category` | где искали: 114 — «Услуги», 0 — весь Авито |
| свойства объявления | `log_price`, `item_rating`, `log_reviews`, `title_length`, `description_length`, `phone_hidden`, `message_forbidden`, `is_service` | цена, рейтинг, число отзывов, длина заголовка и описания, скрыт ли телефон, запрещены ли сообщения, из «Услуг» ли объявление (около 1% корпуса — объявления других категорий, `ingestion/insights.ipynb`). В основную модель не входят, почему — в разделе 5 |

Чтобы увидеть, какие признаки вообще отличают правильные объявления, сравниваем средние значения признаков у правильных и неправильных кандидатов на валидации. Берём только первые 100 кандидатов бейзлайна: так правильные объявления сравниваются с неправильными, которые стоят достаточно высоко и которые реранкеру действительно нужно от них отличать. Число 100 мы не подбирали.

In [4]:
top = val_candidates.filter(pl.col("retrieval_rank") < 100)
positives = top.filter(pl.col("label") == 1)
negatives = top.filter(pl.col("label") == 0)

pl.DataFrame({
    "признак": FEATURES,
    "среднее у правильных": [positives[feature].cast(pl.Float64).mean() for feature in FEATURES],
    "среднее у неправильных": [negatives[feature].cast(pl.Float64).mean() for feature in FEATURES],
})

признак,среднее у правильных,среднее у неправильных
str,f64,f64
"""retrieval_rank""",8.844478,49.881748
"""retrieval_score""",0.723005,0.358989
"""bm25_full""",15.690274,9.237715
"""bm25_title""",7.532538,3.342187
"""bm25_params""",5.467138,3.621213
"""bm25_description""",13.505493,7.946697
"""bm25_filters""",4.11395,3.132317
"""coverage_full""",0.868826,0.59233
"""coverage_title""",0.480045,0.240082


Правильные кандидаты заметно отличаются по признакам текста и локации:

- они выше в выдаче бейзлайна: средний `retrieval_rank` 8.8 против 49.9, то есть, если считать места с единицы, примерно 10-е место против 51-го;
- в них в среднем есть 87% слов запроса против 59%;
- они ближе к месту поиска: 31 км против 168 км;
- их подкатегория чаще совпадает с подкатегорией большинства первых кандидатов: у правильного кандидата в среднем 53% первых 30 кандидатов из той же подкатегории, что и он сам, у неправильного — 34%.

Признаки запроса (`query_max_bm25`, `query_terms`, `has_filters`, `search_category`) у правильных и неправильных почти одинаковые. Так и должно быть: они описывают запрос, а не кандидата, и нужны модели, чтобы по-разному взвешивать остальные признаки для разных запросов. `search_category` в среднем чуть меньше 114, потому что в валидации есть один запрос по всему Авито (раздел 6).

## 3. Добавляем признаки по группам

Учим одну и ту же модель на одних и тех же запросах и добавляем группы признаков по одной, чтобы увидеть, сколько даёт каждая. Начинаем только с места и оценки кандидата в бейзлайне: это почти сам бейзлайн.

Запросы, у которых среди 500 кандидатов нет ни одного правильного объявления, в обучение не идут: на них модели нечему учиться. Отложенные 10% запросов реранкера — те, у которых номер делится на 10. На них после каждого нового дерева считается NDCG@50 — мера качества порядка, которая, как и Recall@50, смотрит на первые 50 мест, но ещё учитывает, насколько высоко среди них стоит правильное объявление. Если 100 деревьев подряд NDCG@50 не растёт, обучение останавливается, и в модели остаются деревья до лучшего значения.

Параметры LightGBM — обычные стартовые значения для такой задачи, мы их не подбирали:

In [5]:
POSITION = ["retrieval_rank", "retrieval_score"]
TEXT = ["bm25_full", "bm25_title", "bm25_params", "bm25_description", "bm25_filters", "coverage_full", "coverage_title", "coverage_params", "coverage_description", "bm25_full_relative", "query_max_bm25"]
LOCATION = ["log_prior", "same_location", "distance_km", "same_location_share"]
MICROCAT_AND_QUERY = ["microcat_share", "query_terms", "has_filters", "search_category"]
FINAL_FEATURES = POSITION + TEXT + LOCATION + MICROCAT_AND_QUERY
assert sorted(FINAL_FEATURES) == sorted(RANKING_FEATURES)

early_stopping_mask = pl.col("query_id").str.slice(1).cast(pl.Int64) % 10 == 0
ranker_train = ranker_candidates.filter(~early_stopping_mask)
ranker_early_stopping = ranker_candidates.filter(early_stopping_mask)

PARAM_MEANINGS = {
    "objective": "чему учится модель: порядку кандидатов внутри запроса (LambdaRank)",
    "n_estimators": "самое большее деревьев; обучение может остановиться раньше",
    "learning_rate": "насколько сильно каждое новое дерево исправляет предыдущие",
    "num_leaves": "самое большее групп, на которые одно дерево делит пары «запрос — кандидат»",
    "min_child_samples": "меньше скольких пар в конечной группе дерева быть не может",
    "subsample": "каждое дерево учится на случайной доле пар",
    "subsample_freq": "случайная доля пар выбирается заново для каждого дерева",
    "colsample_bytree": "каждое дерево видит случайную долю признаков",
    "lambdarank_truncation_level": "сколько первых мест текущего порядка учитывается при обучении",
    "verbose": "не печатать ход обучения",
}
pl.DataFrame({
    "параметр": list(PARAMS),
    "значение": [str(value) for value in PARAMS.values()],
    "что значит": [PARAM_MEANINGS[name] for name in PARAMS],
})

параметр,значение,что значит
str,str,str
"""objective""","""lambdarank""","""чему учится модель: порядку кандидатов внутри запроса (LambdaRank)"""
"""n_estimators""","""2000""","""самое большее деревьев; обучение может остановиться раньше"""
"""learning_rate""","""0.05""","""насколько сильно каждое новое дерево исправляет предыдущие"""
"""num_leaves""","""63""","""самое большее групп, на которые одно дерево делит пары «запрос — кандидат»"""
"""min_child_samples""","""50""","""меньше скольких пар в конечной группе дерева быть не может"""
"""subsample""","""0.8""","""каждое дерево учится на случайной доле пар"""
"""subsample_freq""","""1""","""случайная доля пар выбирается заново для каждого дерева"""
"""colsample_bytree""","""0.8""","""каждое дерево видит случайную долю признаков"""
"""lambdarank_truncation_level""","""100""","""сколько первых мест текущего порядка учитывается при обучении"""


Функции обучения и оценки. Для каждой модели запоминаем Recall@50 на валидации, 95% интервал для него, число признаков и число деревьев в модели.

95% интервал показывает, насколько Recall@50 мог бы сдвинуться, будь проверочные запросы другими. Считаем его так же, как в бейзлайне: 1000 раз случайно выбираем 5 032 проверочных запроса с повторениями, пересчитываем Recall@50 и берём диапазон, в который попали средние 95% результатов (бутстреп).

Число деревьев в модели — то, при котором NDCG@50 на отложенных запросах был лучшим; само обучение шло ещё 100 деревьев дальше, чтобы убедиться, что лучше не станет. Первой строкой в таблицу идёт сам бейзлайн — его первые 50 кандидатов.

In [6]:
results = []
recalls_by_method = {}
models = {}


def evaluate(name: str, predictions: pl.DataFrame, features: int | None = None, trees: int | None = None) -> None:
    recalls = query_recalls(predictions, val)
    low, high = bootstrap_interval(recalls)
    recalls_by_method[name] = recalls
    results.append({
        "модель": name,
        "признаков": features,
        "деревьев в модели": trees,
        "Recall@50": recalls.mean(),
        "95% интервал для Recall@50": f"[{low:.3f}, {high:.3f}]",
    })


def train_and_evaluate(name: str, features: list[str]) -> None:
    model = fit_ranker(ranker_train, ranker_early_stopping, features=features)
    models[name] = model
    evaluate(name, rerank(val_candidates, model.booster_, features=features), len(features), model.best_iteration_)


evaluate("бейзлайн, первые 50", top_by_rank(val_candidates, 50))

In [7]:
train_and_evaluate("позиция в бейзлайне", POSITION)
train_and_evaluate("+ текст", POSITION + TEXT)
train_and_evaluate("+ локация", POSITION + TEXT + LOCATION)
train_and_evaluate("+ подкатегория и запрос", FINAL_FEATURES)
train_and_evaluate("+ свойства объявления", FINAL_FEATURES + ITEM_FEATURES)

pl.DataFrame(results)

модель,признаков,деревьев в модели,Recall@50,95% интервал для Recall@50
str,i64,i64,f64,str
"""бейзлайн, первые 50""",null,null,0.885135,"""[0.876, 0.894]"""
"""позиция в бейзлайне""",2,145,0.880763,"""[0.872, 0.889]"""
"""+ текст""",13,159,0.892488,"""[0.884, 0.901]"""
"""+ локация""",17,131,0.900404,"""[0.893, 0.909]"""
"""+ подкатегория и запрос""",21,221,0.907492,"""[0.900, 0.915]"""
"""+ свойства объявления""",29,252,0.913951,"""[0.907, 0.922]"""


## 4. Не случайны ли улучшения

Интервалы соседних моделей в таблице выше перекрываются, но это ещё не значит, что разница между ними случайна. Все модели проверяются на одних и тех же запросах, и трудные запросы трудны для всех. Поэтому, как в бейзлайне, для каждого проверочного запроса считаем, насколько изменился его Recall@50 при переходе к следующей модели, и строим бутстреп-интервал для средней разницы. В разнице общая для обеих моделей трудность запроса сокращается, поэтому такой интервал уже. Если в него не попадает 0, улучшение не случайное.

In [8]:
def compare(before: str, after: str) -> dict:
    difference = recalls_by_method[after] - recalls_by_method[before]
    low, high = bootstrap_interval(difference)
    return {"было": before, "стало": after, "разница Recall@50": difference.mean(), "95% интервал разницы": f"[{low:.4f}, {high:.4f}]"}


steps = ["бейзлайн, первые 50", "позиция в бейзлайне", "+ текст", "+ локация", "+ подкатегория и запрос", "+ свойства объявления"]
pl.DataFrame([compare(before, after) for before, after in zip(steps, steps[1:])])

было,стало,разница Recall@50,95% интервал разницы
str,str,f64,str
"""бейзлайн, первые 50""","""позиция в бейзлайне""",-0.004372,"""[-0.0078, -0.0012]"""
"""позиция в бейзлайне""","""+ текст""",0.011725,"""[0.0070, 0.0163]"""
"""+ текст""","""+ локация""",0.007916,"""[0.0043, 0.0116]"""
"""+ локация""","""+ подкатегория и запрос""",0.007088,"""[0.0023, 0.0119]"""
"""+ подкатегория и запрос""","""+ свойства объявления""",0.006459,"""[0.0027, 0.0104]"""


Если дать модели только место и оценку кандидата в бейзлайне, Recall@50 даже на 0.4 п.п. ниже, чем у самого бейзлайна (0.885 → 0.881): новой информации у модели нет, и она лишь немного портит его порядок. Дальше каждая группа признаков значимо добавляет своё (интервалы разницы не задевают 0): текст по полям — +1.2 п.п. (0.881 → 0.892), локация — +0.8 (0.892 → 0.900), подкатегория и запрос — +0.7 (0.900 → 0.907).

Модель «+ подкатегория и запрос» (21 признак, Recall@50 = 0.907) дальше называем основной моделью этого ноутбука. Свойства объявления добавляют к ней ещё 0.6 п.п. (0.914), но в неё не входят — почему, в разделе 5.

## 5. Почему свойства объявления не берём

Свойства объявления (цена, рейтинг, отзывы, длины текстов) на валидации помогают, но на тесте могут работать хуже. Проверочные запросы мы взяли из train, поэтому их правильные ответы — всегда объявления, которые кто-то хоть раз выбирал в train. У тестового запроса правильным может оказаться любое объявление корпуса, в том числе ни разу не выбранное в train. Если выбранные в train объявления чем-то отличаются от остальных, модель выучит это отличие, и на валидации оно поможет сильнее, чем на тесте.

Сравниваем число отзывов у трёх групп объявлений. Здесь берём весь train, вместе с проверочными запросами, поэтому правильные ответы валидации входят во вторую группу:

In [9]:
items = ingested.load_items(["item_id", "in_corpus", "item_rating_reviews_count"])
train_item_ids = ingested.load_train(["item_id"]).unique()
corpus_items = items.filter(pl.col("in_corpus")).with_columns(pl.col("item_id").is_in(train_item_ids["item_id"].implode()).alias("in_train"))
val_item_ids = val.select(pl.col("relevant_item_ids").explode(empty_as_null=True).alias("item_id")).unique()
val_items = items.join(val_item_ids, on="item_id", how="semi")

pl.DataFrame({
    "объявления": ["правильные ответы валидации", "объявления корпуса, которые хоть раз выбирали в train", "объявления корпуса, которые ни разу не выбирали в train"],
    "объявлений": [val_items.height, corpus_items.filter(pl.col("in_train")).height, corpus_items.filter(~pl.col("in_train")).height],
    "медиана числа отзывов": [
        val_items["item_rating_reviews_count"].median(),
        corpus_items.filter(pl.col("in_train"))["item_rating_reviews_count"].median(),
        corpus_items.filter(~pl.col("in_train"))["item_rating_reviews_count"].median(),
    ],
})

объявления,объявлений,медиана числа отзывов
str,i64,f64
"""правильные ответы валидации""",4406,42.0
"""объявления корпуса, которые хоть раз выбирали в train""",18142,33.0
"""объявления корпуса, которые ни разу не выбирали в train""",171070,16.0


Объявления корпуса, которые хоть раз выбирали в train, заметно популярнее остальных: медиана числа отзывов 33 против 16, а у правильных ответов валидации — 42. При этом таких объявлений в корпусе только 18 142 из 189 212. Поэтому модель со свойствами объявления может выучить правило «больше отзывов — правильнее», которое на валидации, вероятно, работает лучше, чем на тесте.

Узнать, как эти признаки работают на тесте, можно только по оценке на лидерборде, а вариант со свойствами объявления мы на тесте не проверяли. Поэтому выбираем осторожный вариант: теряем 0.6 п.п. на валидации, зато модель не опирается на популярность объявления.

## 6. Какие признаки важнее всего

Для основной модели (21 признак, без свойств объявления) считаем вклад каждого признака. Каждое дерево много раз делит пары «запрос — кандидат» на две группы по значению одного признака, например «`distance_km` меньше 20». Каждое такое деление уменьшает ошибку модели на обучающих запросах. Вклад признака — на сколько в сумме уменьшилась ошибка по всем делениям по этому признаку. У самих чисел нет понятных единиц, поэтому смотрим на последнюю колонку — долю от суммарного вклада всех признаков.

In [10]:
final = models["+ подкатегория и запрос"]
pl.DataFrame({
    "признак": FINAL_FEATURES,
    "вклад": final.booster_.feature_importance("gain"),
}).with_columns((pl.col("вклад") / pl.col("вклад").sum()).alias("доля вклада")).sort("вклад", descending=True)

признак,вклад,доля вклада
str,f64,f64
"""retrieval_rank""",425043.906981,0.541156
"""retrieval_score""",172457.082251,0.219569
"""microcat_share""",35391.326147,0.045059
"""distance_km""",27753.404352,0.035335
"""bm25_title""",18513.689435,0.023571
"""log_prior""",13206.517774,0.016814
"""same_location_share""",12520.583486,0.015941
"""bm25_full_relative""",12121.382851,0.015433
"""bm25_description""",10078.918007,0.012832


Больше всего модель опирается на место и оценку кандидата в бейзлайне (вместе 76% вклада): в них уже собраны текст и локация. Из новых признаков важнее всего доля подкатегории среди первых 30 кандидатов (4.5%), расстояние до центра локации поиска (3.5%) и BM25 по заголовку (2.4%).

У `search_category` вклад ровно 0. Проверяем, сколько запросов по всему Авито есть в запросах для обучения реранкера и в валидации:

In [11]:
pl.DataFrame({
    "набор": ["обучение реранкера", "валидация"],
    "запросов в «Услугах» (search_category = 114)": [
        ranker_candidates.filter(pl.col("search_category") == 114)["query_id"].n_unique(),
        val_candidates.filter(pl.col("search_category") == 114)["query_id"].n_unique(),
    ],
    "запросов по всему Авито (search_category = 0)": [
        ranker_candidates.filter(pl.col("search_category") == 0)["query_id"].n_unique(),
        val_candidates.filter(pl.col("search_category") == 0)["query_id"].n_unique(),
    ],
})

набор,запросов в «Услугах» (search_category = 114),запросов по всему Авито (search_category = 0)
str,i64,i64
"""обучение реранкера""",6487,0
"""валидация""",5031,1


В запросах для обучения реранкера нет ни одного запроса по всему Авито: у всех `search_category` = 114, признак постоянный, и модель не может им воспользоваться. В валидации такой запрос один из 5 032, а среди тестовых запросов их 9% (`validation/split.ipynb`). Значит, на этих тестовых запросах модель ведёт себя так же, как при поиске в «Услугах», и как она на них работает, валидация не показывает. Признак оставлен, потому что ничему не мешает.

## 7. На каких запросах реранкер помог

Делим проверочные запросы на группы и сравниваем Recall@50 бейзлайна и основной модели. Группы почти те же, что в `retrieval/baseline.ipynb`:

- знаком ли текст: такой же текст искали в train в другом запросе (с другой локацией или фильтрами) или такой текст в train не искали;
- есть ли у запроса фильтры;
- где лежат правильные объявления: все в локации поиска или хотя бы одно в другой локации. Во вторую группу попадают и все запросы, в локации поиска которых нет ни одного объявления корпуса (поиск по региону или стране);
- сколько слов в запросе.

Отличия от бейзлайна: запросы из 2 и 3 слов объединены в одну группу, групп «в локации поиска есть / нет объявлений» здесь нет, а первой строкой добавлены все запросы.

In [12]:
item_locations = ingested.load_items(["item_id", "item_location_id"])
relevant_locations = val.select("query_id", "search_location_id", pl.col("relevant_item_ids").alias("item_id")).explode("item_id", empty_as_null=True)
relevant_locations = relevant_locations.join(item_locations, on="item_id")
relevant_locations = relevant_locations.group_by("query_id").agg(
    (pl.col("item_location_id") == pl.col("search_location_id")).all().alias("relevant_in_search_location")
)

strata = val.select("query_id", "seen_text", "search_infm_params_text", "query_stems")
strata = strata.join(relevant_locations, on="query_id", how="left", maintain_order="left")
strata = strata.with_columns(
    pl.Series("baseline", recalls_by_method["бейзлайн, первые 50"]),
    pl.Series("ranker", recalls_by_method["+ подкатегория и запрос"]),
    pl.col("query_stems").str.split(" ").list.len().alias("words"),
)

groups = {
    "все запросы": pl.lit(True),
    "такой же текст искали в train в другом запросе": pl.col("seen_text"),
    "такой текст в train не искали": ~pl.col("seen_text"),
    "без фильтров": pl.col("search_infm_params_text") == "",
    "с фильтрами": pl.col("search_infm_params_text") != "",
    "все правильные объявления в локации поиска": pl.col("relevant_in_search_location"),
    "хотя бы одно правильное объявление в другой локации": ~pl.col("relevant_in_search_location"),
    "запрос из 1 слова": pl.col("words") == 1,
    "запрос из 2–3 слов": pl.col("words").is_between(2, 3),
    "запрос из 4 слов и больше": pl.col("words") >= 4,
}

pl.DataFrame([
    {
        "группа запросов": name,
        "запросов": strata.filter(condition).height,
        "Recall@50 бейзлайна": strata.filter(condition)["baseline"].mean(),
        "Recall@50 реранкера": strata.filter(condition)["ranker"].mean(),
    }
    for name, condition in groups.items()
])

группа запросов,запросов,Recall@50 бейзлайна,Recall@50 реранкера
str,i64,f64,f64
"""все запросы""",5032,0.885135,0.907492
"""такой же текст искали в train в другом запросе""",1861,0.900233,0.924503
"""такой текст в train не искали""",3171,0.876275,0.897509
"""без фильтров""",2634,0.882055,0.903759
"""с фильтрами""",2398,0.888518,0.911593
"""все правильные объявления в локации поиска""",3899,0.926349,0.938959
"""хотя бы одно правильное объявление в другой локации""",1133,0.743307,0.799206
"""запрос из 1 слова""",192,0.776042,0.796875
"""запрос из 2–3 слов""",3985,0.8867,0.909536


Реранкер улучшает все группы запросов. Сильнее всего — запросы, где хотя бы одно правильное объявление лежит не в локации поиска: 0.74 → 0.80, +5.6 п.п. Вероятно, здесь помогают признаки локации, например расстояние до центра локации поиска, но их вклад отдельно на этой группе мы не проверяли.

Хуже всего после реранкера по-прежнему две группы: эти же запросы и запросы из одного слова, у обеих около 0.80. Однословных запросов всего 192, поэтому их прирост (0.78 → 0.80) может быть случайным: интервалы для отдельных групп мы не считали.

## Итог

Все модели вместе:

In [13]:
pl.DataFrame(results)

модель,признаков,деревьев в модели,Recall@50,95% интервал для Recall@50
str,i64,i64,f64,str
"""бейзлайн, первые 50""",null,null,0.885135,"""[0.876, 0.894]"""
"""позиция в бейзлайне""",2,145,0.880763,"""[0.872, 0.889]"""
"""+ текст""",13,159,0.892488,"""[0.884, 0.901]"""
"""+ локация""",17,131,0.900404,"""[0.893, 0.909]"""
"""+ подкатегория и запрос""",21,221,0.907492,"""[0.900, 0.915]"""
"""+ свойства объявления""",29,252,0.913951,"""[0.907, 0.922]"""


- Среди 500 кандидатов бейзлайна 96.9% правильных объявлений, а в его первых 50 — 88.5%. Из 8.4 п.п. возможного прироста реранкер забирает 2.2: Recall@50 растёт с 0.885 до 0.907, 95% интервал [0.900, 0.915].
- Одних места и оценки бейзлайна модели мало (0.881, на 0.4 п.п. хуже самого бейзлайна): нужны новые признаки. Текст по полям даёт +1.2 п.п., локация +0.8, подкатегория и запрос +0.7, и каждый шаг значим.
- Свойства объявления дают на валидации ещё +0.6 п.п., но в основную модель не входят. Правильные ответы валидации — объявления, которые выбирали в train, а такие объявления популярнее остальных объявлений корпуса (медиана числа отзывов 33 против 16). На тесте этот прирост может не повториться; этот вариант мы на тесте не проверяли.
- Главные признаки после места и оценки бейзлайна — доля подкатегории среди первых 30 кандидатов, расстояние до центра локации поиска и BM25 по заголовку. Запросов по всему Авито в обучении реранкера нет, поэтому такие тестовые запросы (9%) модель не отличает от поиска в «Услугах».
- Реранкер не может поднять 3.1% правильных объявлений, которых нет среди 500 кандидатов. А BM25 находит только объявления, в которых есть слова запроса, но в выборке пар «запрос — выбранное объявление» из train у 4.9% пар в выбранном объявлении нет ни одного слова запроса (`processing/text.ipynb`). Поэтому дальше добавляем второй источник кандидатов — поиск по смыслу: эмбеддер (нейросеть) превращает текст в вектор, и кандидатов ищем по близости векторов запроса и объявления (`embeddings/models.ipynb`). В `ranking/dense.ipynb` реранкер получает этих кандидатов и признаки эмбеддера, и Recall@50 растёт до 0.946.